In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/mnt/lareaulab/reliscu/code")

from parse_gtf import *

In [5]:
psi_data = "SyntheticDataset1_20pcntCells_25SD_200samples_SJ_pseudobulk_PSI"

In [ ]:
psi = pd.read_csv(f"data/yao_2021_MOp_STAR_{psi_data}.csv", index_col=0)
top_qval_mods_df = pd.read_csv("data/enrichments/yao_2021_MOp_pairwise_DE_genes_dream_uniqueFALSE_20pcntCells_25SD_200samples_pseudobulk_mergeParam0.85_PosBC_top_Qval_modules.csv")

### Add gene names to PSI data

In [8]:
# Parse GTF attribute column
gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCm39/gencode.vM35.annotation.gtf"
gtf = gtf_parse(gtf_file)
gtf_subset = gtf.loc[gtf['feature'].isin(["gene"])]
attrs = gtf_subset["attribute"].apply(extract_attributes)
attrs_df = attrs.apply(pd.Series)
gtf_parsed = pd.concat([gtf_subset.drop(columns=["attribute"]), attrs_df], axis=1)

In [9]:
# Get PSI and GTF data ready to merge on gene IDs
gtf_parsed['gene_id'] = gtf_parsed['gene_id'].str.split(".").str[0]
psi['gene_id'] = psi.index.str.split("_").str[0]
psi['exon_id'] = psi.index.values

In [10]:
psi_anno = pd.merge(gtf_parsed[['gene_id', 'gene_name']], psi, on="gene_id", how="right")
psi_anno = psi_anno.set_index("exon_id").rename_axis(None)
psi_anno = psi_anno.drop(columns=["gene_id"])

### Calc. corr between ME and exon PSI

In [11]:
corr_df = pd.DataFrame(
    columns=["Gene"] + top_qval_mods_df['Cell_type'].tolist(), 
    index=psi_anno.index
)
corr_df['Gene'] = psi_anno['gene_name'] 

for i, row in top_qval_mods_df.iterrows():
    ctype = row['Cell_type']

    mod_df = pd.read_csv(row['ME_path'])
    mod_eig = mod_df.set_index("Sample")[row['Module']]
    mod_eig = pd.to_numeric(mod_eig, errors="coerce")
    
    corrs = psi_anno.iloc[:, 1:].corrwith(mod_eig, axis=1)
    corr_df[ctype] = corrs

In [12]:
corr_df.head()

,Gene,endothelial_cell,oligodendrocyte,pvalb_GABAergic_cortical_interneuron,L4_5_intratelencephalic_projecting_glutamatergic_neuron_of_the_primary_motor_cortex,L5_6_near-projecting_glutamatergic_neuron_of_the_primary_motor_cortex,sst_chodl_GABAergic_cortical_interneuron,L6_corticothalamic-projecting_glutamatergic_cortical_neuron,VIP_GABAergic_cortical_interneuron,lamp5_GABAergic_cortical_interneuron,sncg_GABAergic_cortical_interneuron,L6b_glutamatergic_cortical_neuron,L6_intratelencephalic_projecting_glutamatergic_neuron_of_the_primary_motor_cortex,L2_3-6_intratelencephalic_projecting_glutamatergic_neuron,L5_extratelencephalic_projecting_glutamatergic_cortical_neuron,sst_GABAergic_cortical_interneuron
ENSMUSG00000033845_ProteinCoding_1,Mrpl15,0.110098,0.052936,-0.271649,0.184399,0.055633,-0.052832,0.171403,-0.036733,0.004798,-0.000776,0.173564,-0.047883,0.070007,-0.189439,-0.221400
ENSMUSG00000025903_ProteinCoding_1,Lypla1,-0.017629,-0.027522,0.004350,-0.079437,0.006498,0.011708,0.054122,-0.038815,0.086095,-0.143246,0.106559,0.008953,-0.020263,-0.022257,0.020005
ENSMUSG00000025903_ProteinCoding_2,Lypla1,0.015606,0.066510,-0.057634,0.018534,-0.064982,0.038789,-0.061600,0.019479,-0.131806,0.065543,-0.044573,0.014161,0.047299,0.045232,0.112769
ENSMUSG00000002459_ProteinCoding_1,Rgs20,0.069856,-0.036614,-0.090042,0.057108,-0.063172,0.042940,0.013753,0.068285,0.031633,-0.000250,-0.012551,0.041200,0.022470,-0.084701,-0.076993
ENSMUSG00000033793_ProteinCoding_1,Atp6v1h,0.017506,0.002138,-0.032288,-0.059212,0.025356,-0.043745,0.138843,-0.075247,-0.027349,-0.154536,0.036795,0.258574,-0.045546,-0.145179,-0.108774


In [24]:
corr_df.sort_values("pvalb_GABAergic_cortical_interneuron", ascending=False)[['Gene', 'pvalb_GABAergic_cortical_interneuron']].head(n=20)

,Gene,pvalb_GABAergic_cortical_interneuron
ENSMUSG00000059534_other_1,Uqcr10,0.804130
ENSMUSG00000068739_ProteinCoding_1,Sars1,0.791331
ENSMUSG00000026797_ProteinCoding_1,Stxbp1,0.747566
ENSMUSG00000020436_ProteinCoding_1,Gabrg2,0.723426
ENSMUSG00000022537_NMD_2,Tmem44,0.674515
ENSMUSG00000060261_ProteinCoding_6,Gtf2i,0.671145
ENSMUSG00000033530_other_1,Ttc7b,0.646264
ENSMUSG00000026825_ProteinCoding_1,Dnm1,0.628011
ENSMUSG00000026150_ProteinCoding_7,Mff,0.600021
ENSMUSG00000027634_ProteinCoding_1,Ndrg3,0.596532


In [19]:
corr_df.sort_values("Vip", ascending=False)

,Gene,SMC-Peri,Pvalb,Lamp5,L2_3_IT,L6b,L6_CT,L4_5_IT,Vip,L5_6_NP,Sst_Chodl,L6_IT,Sncg,L5_PT,L5_IT
ENSMUSG00000052726_ProteinCoding_6,Kcnt2,-0.002990,-0.005626,0.129550,-0.073296,-0.238808,-0.138139,-0.112924,0.809340,-0.023628,0.226871,-0.217225,0.308322,0.054335,-0.436236
ENSMUSG00000033768_ProteinCoding_8,Nrxn2,0.053666,0.119016,0.081018,-0.087985,-0.076015,-0.173873,-0.276618,0.776522,0.008305,0.436359,-0.299793,0.426461,0.041548,-0.590581
ENSMUSG00000024617_ProteinCoding_1,Camk2a,-0.021899,0.257328,0.205088,-0.216194,0.000648,-0.157216,-0.317337,0.751652,-0.022144,0.394860,-0.371998,0.341149,-0.001789,-0.543603
ENSMUSG00000023092_ProteinCoding_2,Fhl1,-0.044391,0.066027,0.050187,-0.339433,0.149514,-0.008909,-0.138105,0.715969,0.279123,0.045492,-0.352220,0.355323,-0.010057,-0.301109
ENSMUSG00000060424_other_9,Pantr1,0.135284,-0.045746,0.083165,0.040383,0.048118,-0.351894,0.290980,0.707624,-0.172355,-0.014587,-0.381689,0.431195,-0.043029,-0.365723
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSMUSG00000021124_ProteinCoding_2,Vti1b,0.030700,-0.282945,-0.203990,0.171619,0.001209,0.395240,0.124260,-0.711246,-0.049353,-0.480896,0.469005,-0.405557,-0.074037,0.549764
ENSMUSG00000066392_ProteinCoding_2,Nrxn3,-0.066444,0.150183,-0.111569,0.127928,-0.050957,0.121709,0.109676,-0.748577,0.053768,0.038503,0.121018,-0.816708,0.000198,0.349015
ENSMUSG00000028399_ProteinCoding_8,Ptprd,0.042013,-0.260369,-0.393637,0.059030,0.019570,0.228817,0.351912,-0.762786,0.164100,-0.259606,0.193309,-0.530223,-0.026857,0.611066
ENSMUSG00000024109_ProteinCoding_3,Nrxn1,-0.022560,-0.027453,-0.427096,0.062040,0.189734,0.169366,0.106837,-0.801975,0.177852,-0.085501,0.251774,-0.648137,-0.001460,0.465353


In [ ]:
corr_df.to_csv(f"data/yao_2021_MOp_STAR_{psi_data}_exon_corr.csv")